# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a dataset described by a [Croissant Schema](https://mlcommons.org/croissant/) using the `mlcroissant` library. The dataset covers mass spectrometry data for extracellular vesicles isolated from human mast cells and includes quantitative protein information.

### Dataset Source
The dataset schema is provided at the following Croissant JSON-LD URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(url)

# View general metadata
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}\n")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview

List available record sets and their properties. Each record set, field, and column is referenced by its `@id`.

In [ ]:
# List all record sets defined by their @id and display their field @id and names
record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f" ▶ Record set ID: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"   Name: {rs.name}")
    print(f"   Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id} | Name: {f.name}")
    print()

## 3. Data Extraction

We'll extract data from the main record sets. All `@id`s are used as required by the Croissant specification and for downstream referencing.

Replace `your_record_set_id` with the desired `@id` identified above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

if record_set_ids:
    print(f"\nColumns of first record set ({record_set_ids[0]}):")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

This section explores the data frame: filtering, normalizing, grouping data, and other typical analysis steps.

**Note:** Replace placeholders with the specific field `@id`s from displayed columns. In the mass spectrometry dataset, numeric fields may include quantities like normalized abundance, coverage, peptide counts, molecular weight, etc.

In [ ]:
# Select an appropriate record set and numeric field by @id
# For demonstration, we take the first record set if available
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}\n")
    print("Sample columns:", df.columns.tolist())

    # Example: Pick the first numeric field from available columns
    # You can replace this field with another appropriate numeric field @id
    import numpy as np

    # Try to infer a numeric column (simplified heuristic)
    numeric_field = None
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype != object:
                numeric_field = col
                break
        except:
            continue
    
    if numeric_field is not None:
        threshold = df[numeric_field].mean()

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {filtered_df.shape[0]} records\n")

        filtered_df[numeric_field + "_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )

        print(f"Top 5 filtered records with normalized {numeric_field}:")
        display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Try grouping by another field (e.g., the first categorical column)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                if df[col].nunique() < df.shape[0] and df[col].nunique() < 20:
                    group_field = col
                    break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print('No numeric field found to analyze.')
else:
    print("No record sets available for analysis.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and, if grouped, the means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution if analysis succeeded above
if 'filtered_df' in locals() and numeric_field in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If we have a group_field, show mean by group
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8, 4))
        grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        sns.barplot(y=grouped.index, x=grouped.values, palette="viridis")
        plt.title(f"Mean {numeric_field} by {group_field} (filtered)")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated:
- Loading mass spectrometry dataset metadata and records with `mlcroissant`.
- Discovering available record sets and their field `@id`s.
- Extracting and exploring tabular data from the Croissant package, referencing all entities by their `@id`.
- Applying typical EDA steps: filtering, normalizing, grouping, and visualizing protein-level data.

This workflow enables automated and transparent exploration of FAIRML datasets described by Croissant schemas. For further analysis, refer to field and record set `@id`s as needed and extend the notebook with deeper biological/statistical exploration.